In [1]:
import subprocess
import json

In [4]:
class LEANRepl :

    def __init__(self) :
        """ 
            Starts the LEANRepfl as a subprocess. Once import_project is called proofs can be verified on the subprocess
        """
        self.proc = subprocess.Popen(["lake", "env", "../repl_minif2f/.lake/build/bin/repl"],
                        cwd="/Users/eduardo/Desktop/LEAN_AI/miniF2F", stdin=subprocess.PIPE, stdout=subprocess.PIPE, text = True)
        self.base_env = None

    def send(self, message):
        """
            writes the message inside the LEAN environement and returns the LEAN evaluation
            - Used only for import_project and auxiliar function in write_proof -
            - writes the message like it is, no json dumps inside -
        """
        #writing of the message
        self.proc.stdin.write(message)
        self.proc.stdin.flush()

        # reading of LEAN answer
        lines = []
        line = self.proc.stdout.readline()
        while(line != "\n") :
            if(line == "") :
                print("error on LEAN answer!!")
                break
            lines.append(line)  # the JSON response
            line = self.proc.stdout.readline()
        return json.loads("".join(lines))


    def write_proof(self, cmd):
        """
            writes the proof cmd inside the LEAN environement and returns the LEAN evaluation
            cmd is a string with the proof
        """
        #checking import_project was run
        if(self.base_env == None):
            raise RuntimeError("didn't calles import_project before write_proof")
        #sending the proof and returning LEAN answer
        return self.send(json.dumps({"cmd": cmd, "env": self.base_env}) + "\n\n")
        


    def import_project(self):
        """
            Imports MiniF2F.ProblemImports, needed for every proof and saves time by doing it just once
        """
        response = self.send(json.dumps({"cmd": "import MiniF2F.ProblemImports\n"}) + "\n\n")
        self.base_env = response["env"]
        return response

    def close(self):
        """
            closes the subprocess, to be done after every LEANRepl class served it use
        """
        self.proc.stdin.close()
        try:
            self.proc.wait(5)
        except subprocess.TimeoutExpired:
            self.proc.kill()

    def check(self, answer):
        """
            checks if LEAN considers the proof correct,incorrect or if there's the use of sorry in the proof

            param:
                answer: the LEAN evaluation returned by write_proof((self, cmd))

            returns:
                1 if the proof is correct, 0 if it uses sorry, -1 if there's an error
        """

        ########what if there's warnings like unused variable but the prrof is correct? To check
        if("messages" not in answer and "sorries" not in answer):
            return 1
        if("messages" in answer):
            if(any(m["severity"] == "error" for m in answer["messages"])):
                return -1
            elif("sorries" in answer):
                return 0
            else:
                return 1
        if("sorries" in answer):
            return 0
        
        raise RuntimeError(f"No case for {answer}")

In [5]:
Repl = LEANRepl()
Repl.import_project()

{'env': 0}

In [7]:
cmd1 = "example : (2:ℕ) + 2 = 4 := by rfl"
answer1 = Repl.write_proof(cmd1)
check1 = Repl.check(answer1)

check1

1

/-- Expand the following expression: $7(3y+2)$ -/
theorem mathd_algebra_182 (y : ℂ) : 7 * (3 * y + 2) = answer(21 * y + 14) := by
  sorry 

- take out all comments : /-- ... --/
- don't mind for retours à la ligne

In [15]:
cmd2 = "theorem mathd_algebra_182 (y : ℂ) : 7 * (3 * y + 2) = answer(21 * y \n + 14) := \n by sorry"
answer2 = Repl.write_proof(cmd2)
check2 = Repl.check(answer2)

check2

0

In [18]:
ignore_comment():
if /- , while not -/ skip 

treat_header():
""" to treat all the import and open beginning of the file"""

split_theorems():

SyntaxError: invalid syntax (3514109581.py, line 1)

In [ ]:
Repl.close()